In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Input


# ----------------------------
# 1. Corpus pequeño (10 oraciones en español)
# ----------------------------
sentences = [
    "la vida es color de rosa",
    "la clase de PLN es muy buena y divertida",
    "la ciudad de Cali es muy bonita",
    "la Universidad del Valle es una universidad pública",
    "el perro corre por el parque",
    "el gato duerme sobre la mesa",
    "me gusta aprender modelos de lenguaje",
    "las redes neuronales aprenden patrones de datos",
    "el estudiante lee un libro de historia",
    "hoy hace mucho calor en la ciudad"
]
print(f"Total de oraciones cargadas: {len(sentences)}")

# ----------------------------
# 2. Tokenización
# ----------------------------
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1
print(f"Vocabulario: {vocab_size} palabras")

sequences = tokenizer.texts_to_sequences(sentences)

# ----------------------------
# 3. Pares (contexto → siguiente palabra)
# ----------------------------
X, y = [], []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

print(f"Total de ejemplos generados: {len(X)}")

# ----------------------------
# 4. Padding
# ----------------------------
MAX_LEN = 10  # corpus pequeño → longitud corta
X_padded = pad_sequences(X, maxlen=MAX_LEN, padding='pre')
y = np.array(y)

# ----------------------------
# 5. Split train/test
# ----------------------------
split = int(0.8 * len(X_padded))
X_train, X_test = X_padded[:split], X_padded[split:]
y_train, y_test = y[:split], y[split:]

# ----------------------------
# 6. Modelo BiLSTM
# ----------------------------
EMBEDDING_DIM = 200
LSTM_UNITS = 128

model = Sequential([
    Input(shape=(MAX_LEN,)),                       # ← define la forma de entrada
    Embedding(vocab_size, EMBEDDING_DIM),
    Bidirectional(LSTM(LSTM_UNITS)),
    Dense(32, activation='relu'),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

# ----------------------------
# 7. Entrenamiento
# ----------------------------
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
EPOCHS = 50
BATCH_SIZE = 16

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=2
)

# ----------------------------
# 8. Predicción de la siguiente palabra
# ----------------------------
def predict_next_word(model, tokenizer, sentence, max_len):
    sequence = tokenizer.texts_to_sequences([sentence])[0]
    if len(sequence) == 0:
        return "<no_reconocido>"
    sequence_padded = pad_sequences([sequence], maxlen=max_len, padding='pre')
    prediction = model.predict(sequence_padded, verbose=0)
    predicted_idx = np.argmax(prediction[0])
    return tokenizer.index_word.get(predicted_idx, "<desconocido>")

# ----------------------------
# 9. Interfaz interactiva
# ----------------------------
print("\n" + "="*60)
print("🔮 PREDICCIÓN DE SIGUIENTE PALABRA (corpus pequeño)")
print("Escribe 'salir' para terminar.")
print("="*60)

while True:
    user_input = input("\n📌 Tu frase: ").strip()
    if user_input.lower() in ["salir", "exit", "quit"]:
        print("👋 Hasta luego.")
        break
    if not user_input:
        continue
    next_word = predict_next_word(model, tokenizer, user_input, MAX_LEN)
    print(f"➡️  Predicción: '{next_word}'")

Total de oraciones cargadas: 10
Vocabulario: 52 palabras
Total de ejemplos generados: 59


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 10, 200)        │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       336,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 52)             │         1,716 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 357,236 (1.36 MB)

 Trainable params: 357,236 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
3/3 - 4s - 1s/step - accuracy: 0.0000e+00 - loss: 3.9506 - val_accuracy: 0.2000 - val_loss: 3.9873
Epoch 2/50
3/3 - 0s - 52ms/step - accuracy: 0.0714 - loss: 3.9303 - val_accuracy: 0.2000 - val_loss: 4.0197
Epoch 3/50
3/3 - 0s - 101ms/step - accuracy: 0.0238 - loss: 3.9050 - val_accuracy: 0.0000e+00 - val_loss: 4.0656
Epoch 4/50
3/3 - 0s - 55ms/step - accuracy: 0.0476 - loss: 3.8658 - val_accuracy: 0.2000 - val_loss: 4.1334
Epoch 5/50
3/3 - 0s - 70ms/step - accuracy: 0.0952 - loss: 3.8252 - val_accuracy: 0.2000 - val_loss: 4.2646
Epoch 6/50
3/3 - 0s - 128ms/step - accuracy: 0.0952 - loss: 3.7503 - val_accuracy: 0.2000 - val_loss: 4.4730
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.

🔮 PREDICCIÓN DE SIGUIENTE PALABRA (corpus pequeño)
Escribe 'salir' para terminar.
➡️  Predicción: 'calor'
➡️  Predicción: 'muy'
➡️  Predicción: 'calor'
➡️  Predicción: 'calor'
➡️  Predicción: 'muy'
➡️  Predicción: 'calor'
➡️  Predicción: 'calor'

📌 Tu frase: l